# 🧠 step2_train.py — EfficientNetB0 Transfer Learning
**KrishiAI · IT-A · 2024-25**

### Architecture
```
Input (224×224×3)
  → EfficientNetB0 (pretrained ImageNet)
  → GlobalAveragePooling2D
  → BatchNormalization → Dropout(0.3)
  → Dense(256, ReLU, L2 regularization)
  → BatchNormalization → Dropout(0.3)
  → Dense(7, Softmax)          ← 7 Udupi disease classes
```

### Two-Phase Training
| Phase | What trains | Learning Rate | Epochs | Purpose |
|-------|-------------|--------------|--------|---------|
| **1** | Head only (base frozen) | 1e-3 | 10 | Fast feature learning |
| **2** | Head + top 30 base layers | 1e-5 | 20 | Fine-tune on crop diseases |

> ⚠️ **Run `02_step1_data_pipeline.ipynb` first** — this notebook loads the manifests it saves.  
> ⏱️ Expected: ~10 min Phase 1 + ~25 min Phase 2 on Colab T4 GPU.

In [ ]:
# Install required packages (pre-installed on Colab; run once locally)
!pip install tensorflow==2.13.0 tensorflow-datasets==4.9.2 --quiet
!pip install opencv-python-headless Pillow tqdm seaborn scikit-learn --quiet
print("✅ Packages ready")

In [ ]:
import os, sys, json, time
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard, CSVLogger
)
tf.random.set_seed(42)
np.random.seed(42)
print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ── config.py is the shared settings file — paste it here so this notebook
# ── is self-contained. All other notebooks also start with this block.
# =============================================================================
# config.py — KrishiAI Project Configuration
# All hyperparameters, paths, and class definitions in ONE place.
# Change values here; the rest of the scripts pick them up automatically.
# =============================================================================

import os

# ── PATHS ──────────────────────────────────────────────────────────────────────
BASE_DIR        = os.getcwd()   # notebook working directory
DATA_RAW_DIR    = os.path.join(BASE_DIR, "data", "raw")          # downloaded zips
DATA_CLEAN_DIR  = os.path.join(BASE_DIR, "data", "cleaned")      # after cleaning
DATA_PROC_DIR   = os.path.join(BASE_DIR, "data", "processed")    # train/val/test splits
MODELS_DIR      = os.path.join(BASE_DIR, "models")               # saved .keras / .h5
TFLITE_DIR      = os.path.join(BASE_DIR, "tflite")               # exported .tflite
LOGS_DIR        = os.path.join(BASE_DIR, "logs")                 # TensorBoard / CSVs
PLOTS_DIR       = os.path.join(BASE_DIR, "plots")                # confusion matrix, curves

for d in [DATA_RAW_DIR, DATA_CLEAN_DIR, DATA_PROC_DIR,
          MODELS_DIR, TFLITE_DIR, LOGS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── DATASET ───────────────────────────────────────────────────────────────────
# PlantVillage is available via tensorflow_datasets (no Kaggle key needed).
# We filter it down to only classes relevant to Udupi crops.
# The keys below MUST match the class folder names in PlantVillage exactly.

DISEASE_CLASSES = {
    # ── Rice ──────────────────────────────────────────────────────────────────
    "Rice___Leaf_blast":          "Rice Blast",            # Magnaporthe oryzae
    "Rice___Bacterial_leaf_blight": "Bacterial Leaf Blight", # Xanthomonas oryzae
    "Rice___Brown_spot":          "Brown Spot",            # Bipolaris oryzae
    "Rice___healthy":             "Healthy Rice",

    # ── Coconut / proxy (PlantVillage doesn't have coconut; we use the
    #    closest abiotic-stress proxy — corn with nutrient deficiency)
    "Corn_(maize)___Northern_Leaf_Blight": "Leaf Blight (proxy)",
    "Corn_(maize)___healthy":              "Healthy Crop (proxy)",

    # ── Tomato (used as augmentation source for fungal patterns) ──────────────
    "Tomato___Late_blight":       "Fungal Blight (proxy)",
    "Tomato___healthy":           "Healthy Crop (proxy)",
}

# Friendly short labels for plotting (must stay in same order as CLASS_NAMES list)
CLASS_NAMES = [
    "Rice Blast",
    "Bacterial Blight",
    "Brown Spot",
    "Healthy Rice",
    "Leaf Blight",
    "Fungal Blight",
    "Healthy Crop",
]

NUM_CLASSES = len(CLASS_NAMES)

# ── MODEL ─────────────────────────────────────────────────────────────────────
IMAGE_SIZE   = (224, 224)   # EfficientNetB0 native input size
BATCH_SIZE   = 32
EPOCHS_FROZEN  = 10        # Phase 1: train only the classification head
EPOCHS_FINETUNE = 20       # Phase 2: unfreeze top layers and fine-tune
LEARNING_RATE_HEAD    = 1e-3
LEARNING_RATE_FINETUNE = 1e-5
DROPOUT_RATE = 0.3
L2_LAMBDA    = 1e-4

# ── DATA SPLIT ────────────────────────────────────────────────────────────────
TRAIN_SPLIT = 0.70
VAL_SPLIT   = 0.15
TEST_SPLIT  = 0.15
RANDOM_SEED = 42

# ── DATA CLEANING THRESHOLDS ──────────────────────────────────────────────────
MIN_IMAGE_SIZE_BYTES = 2_000        # discard images smaller than 2 KB (corrupt)
MIN_PIXEL_DIMENSION  = 64           # discard images narrower/shorter than 64px
MAX_BLUR_VARIANCE    = 10.0         # discard images with Laplacian variance < 10
                                    # (too blurry to be useful)

# ── AUGMENTATION ──────────────────────────────────────────────────────────────
# Tuned for monsoon / field conditions in Karnataka
AUGMENT_CONFIG = {
    "rotation_range":     20,        # ±20° for handheld phone shots
    "zoom_range":         0.2,       # simulate distance variation
    "horizontal_flip":    True,
    "brightness_range":   (0.7, 1.3),# monsoon overcast → harsh afternoon sun
    "width_shift_range":  0.1,
    "height_shift_range": 0.1,
    "shear_range":        0.1,
    "fill_mode":          "reflect",
}

# ── TFLITE EXPORT ─────────────────────────────────────────────────────────────
TFLITE_MODEL_NAME  = "krishiai_efficientnetb0_int8.tflite"
TFLITE_TARGET_MB   = 4.0   # must stay under 4 MB for ₹8k Android phones

# ── KANNADA MESSAGES ──────────────────────────────────────────────────────────
# Shown in the inference output alongside the English diagnosis
KANNADA_ALERTS = {
    "Rice Blast":          "ಭತ್ತದ ಬ್ಲಾಸ್ಟ್ ರೋಗ ಪತ್ತೆ. ಟ್ರೈಸೈಕ್ಲಾಜ಼ೋಲ್ ತಕ್ಷಣ ಸಿಂಪಡಿಸಿ.",
    "Bacterial Blight":    "ಬ್ಯಾಕ್ಟೀರಿಯಲ್ ಎಲೆ ಒಣಗುವಿಕೆ. ಕಾಪರ್ ಆಕ್ಸಿಕ್ಲೋರೈಡ್ ಬಳಸಿ.",
    "Brown Spot":          "ಕಂದು ಚುಕ್ಕೆ ರೋಗ. ಮ್ಯಾಂಕೋಜ಼ೆಬ್ ಬಳಸಿ.",
    "Healthy Rice":        "ಬೆಳೆ ಆರೋಗ್ಯಕರವಾಗಿದೆ. ಮೇಲ್ವಿಚಾರಣೆ ಮುಂದುವರಿಸಿ.",
    "Leaf Blight":         "ಎಲೆ ಒಣಗುವ ರೋಗ. KVK ಸಂಪರ್ಕಿಸಿ.",
    "Fungal Blight":       "ಶಿಲೀಂಧ್ರ ರೋಗ ಪತ್ತೆ. ಶಿಲೀಂಧ್ರನಾಶಕ ಸಿಂಪಡಿಸಿ.",
    "Healthy Crop":        "ಬೆಳೆ ಆರೋಗ್ಯಕರ. ನಿಯಮಿತ ತಪಾಸಣೆ ಮಾಡಿ.",
}

TREATMENT_MAP = {
    "Rice Blast":       ["Tricyclazole 75WP @ 0.6 g/L", "Isoprothiolane 40EC @ 1.5 mL/L", "Avoid excess nitrogen", "Drain stagnant water"],
    "Bacterial Blight": ["Copper Oxychloride 50WP @ 3 g/L", "Streptocycline @ 0.5 g/L", "Avoid flood irrigation", "Remove infected debris"],
    "Brown Spot":       ["Mancozeb 75WP @ 2.5 g/L", "Propiconazole 25EC @ 1 mL/L", "Correct potassium deficiency"],
    "Healthy Rice":     ["Continue regular monitoring", "Preventive fungicide (optional at tillering)"],
    "Leaf Blight":      ["Copper-based fungicide", "Consult KVK Udupi: 0820-2520842"],
    "Fungal Blight":    ["Mancozeb + Carbendazim combo", "Improve field drainage"],
    "Healthy Crop":     ["Continue monitoring", "Check weekly during monsoon"],
}


## 📚 Function Definitions

In [ ]:
# =============================================================================
# step2_train.py — Build EfficientNetB0 + Train in Two Phases
#
# Architecture:
#   EfficientNetB0 (pretrained ImageNet) → GlobalAveragePooling → Dropout
#   → Dense(256, relu) → Dropout → Dense(NUM_CLASSES, softmax)
#
# Training strategy (standard transfer learning):
#   Phase 1  — Freeze EfficientNetB0 base; train only the new head (10 epochs)
#   Phase 2  — Unfreeze top ~30 layers of base; fine-tune at very low LR (20 epochs)
#
# Run: python step2_train.py
# =============================================================================

import os, sys, json, time

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint,
    TensorBoard, CSVLogger
)
import matplotlib
# (inline mode set by %matplotlib inline)
import matplotlib.pyplot as plt


# ── Reproducibility ───────────────────────────────────────────────────────────
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"TensorFlow {tf.__version__}  |  GPU: {tf.config.list_physical_devices('GPU')}")


# ──────────────────────────────────────────────────────────────────────────────
# REBUILD DATASETS FROM SAVED MANIFESTS
# (So step2 can run independently from step1 once data is prepared.)
# ──────────────────────────────────────────────────────────────────────────────

def load_datasets_from_manifests():
    """Load the train/val/test manifests saved by step1 and rebuild tf.data."""
    AUTOTUNE = tf.data.AUTOTUNE

    def pairs_from_csv(name):
        df = pd.read_csv(os.path.join(LOGS_DIR, f"{name}_manifest.csv"))
        return list(zip(df["path"].tolist(), df["label"].tolist()))

    def make_ds(pairs, augment=False, shuffle=False):
        paths  = [p for p, _ in pairs]
        labels = [l for _, l in pairs]
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if shuffle:
            ds = ds.shuffle(len(pairs), seed=RANDOM_SEED, reshuffle_each_iteration=True)
        ds = ds.map(lambda p, l: _load_image(p, l, augment), num_parallel_calls=AUTOTUNE)
        ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
        return ds, len(pairs)

    train_ds, n_train = make_ds(pairs_from_csv("train"), augment=True,  shuffle=True)
    val_ds,   n_val   = make_ds(pairs_from_csv("val"),   augment=False, shuffle=False)
    test_ds,  n_test  = make_ds(pairs_from_csv("test"),  augment=False, shuffle=False)

    print(f"  train={n_train}  val={n_val}  test={n_test}")
    return train_ds, val_ds, test_ds


@tf.function
def _load_image(path, label, augment):
    """Load, decode, preprocess (EfficientNet expects [0,255] float32)."""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.cast(img, tf.float32)   # EfficientNet preprocessor handles normalisation

    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)                # rotation proxy
        img = tf.image.random_brightness(img, max_delta=0.25 * 255)
        img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
        img = tf.image.random_saturation(img, lower=0.7, upper=1.3)
        img = tf.image.random_hue(img, max_delta=0.05)
        img = tf.clip_by_value(img, 0.0, 255.0)

    # EfficientNet built-in preprocessor (scales to [-1, 1] internally)
    img = preprocess_input(img)
    label_one_hot = tf.one_hot(label, NUM_CLASSES)
    return img, label_one_hot


# ──────────────────────────────────────────────────────────────────────────────
# MODEL ARCHITECTURE
# ──────────────────────────────────────────────────────────────────────────────

def build_model(num_classes: int = NUM_CLASSES, trainable_base: bool = False):
    """
    Build the KrishiAI classification model.

    Architecture:
        Input (224x224x3)
        → EfficientNetB0 base (pretrained, optionally frozen)
        → GlobalAveragePooling2D          # reduces spatial dims
        → BatchNormalization
        → Dropout(0.3)
        → Dense(256, relu, L2)            # custom head
        → BatchNormalization
        → Dropout(0.3)
        → Dense(NUM_CLASSES, softmax)     # disease probability

    Args:
        trainable_base: If False (Phase 1), all EfficientNetB0 weights are frozen.
                        If True  (Phase 2), top layers are unfrozen for fine-tuning.
    """
    # ── Base model ────────────────────────────────────────────────────────────
    base = EfficientNetB0(
        weights="imagenet",          # use ImageNet pretrained weights
        include_top=False,           # remove ImageNet classification head
        input_shape=(*IMAGE_SIZE, 3)
    )

    if not trainable_base:
        # Phase 1: Freeze the entire base
        base.trainable = False
        print("  Base frozen (Phase 1 — head-only training)")
    else:
        # Phase 2: Unfreeze only the top ~30 layers
        base.trainable = True
        for layer in base.layers[:-30]:
            layer.trainable = False
        trainable_count = sum(1 for l in base.layers if l.trainable)
        print(f"  Base partially unfrozen ({trainable_count}/{len(base.layers)} layers trainable, Phase 2)")

    # ── Build full model with Functional API ─────────────────────────────────
    inputs = keras.Input(shape=(*IMAGE_SIZE, 3), name="crop_image")

    # Feature extraction
    x = base(inputs, training=False)          # training=False keeps BN frozen in Phase 1
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.Dropout(DROPOUT_RATE, name="drop1")(x)

    # Classification head
    x = layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=regularizers.l2(L2_LAMBDA),
        name="dense_head"
    )(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.Dropout(DROPOUT_RATE, name="drop2")(x)

    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    model = keras.Model(inputs, outputs, name="KrishiAI_EfficientNetB0")
    return model, base


# ──────────────────────────────────────────────────────────────────────────────
# CALLBACKS
# ──────────────────────────────────────────────────────────────────────────────

def get_callbacks(phase: int) -> list:
    """
    Returns the list of Keras callbacks for a given training phase.
    phase=1 → head training | phase=2 → fine-tuning
    """
    checkpoint_path = os.path.join(MODELS_DIR, f"best_phase{phase}.keras")

    callbacks = [
        # Save the best model by validation accuracy
        ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_accuracy",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),
        # Reduce learning rate if val_loss plateaus
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
        # Stop early if val_loss doesn't improve for 7 epochs
        EarlyStopping(
            monitor="val_loss",
            patience=7,
            restore_best_weights=True,
            verbose=1
        ),
        # Log metrics to CSV
        CSVLogger(
            os.path.join(LOGS_DIR, f"phase{phase}_training_log.csv"),
            append=False
        ),
        # TensorBoard logs
        TensorBoard(
            log_dir=os.path.join(LOGS_DIR, f"tensorboard_phase{phase}"),
            histogram_freq=1
        ),
    ]
    return callbacks, checkpoint_path


# ──────────────────────────────────────────────────────────────────────────────
# LOAD CLASS WEIGHTS
# ──────────────────────────────────────────────────────────────────────────────

def load_class_weights():
    path = os.path.join(LOGS_DIR, "class_weights.json")
    if os.path.exists(path):
        with open(path) as f:
            raw = json.load(f)
        return {int(k): v for k, v in raw.items()}
    else:
        print("  [WARN] class_weights.json not found — using uniform weights")
        return None


# ──────────────────────────────────────────────────────────────────────────────
# PLOT TRAINING HISTORY
# ──────────────────────────────────────────────────────────────────────────────

def plot_history(history, phase_label: str, filename: str):
    """Save loss + accuracy curves for one training phase."""
    epochs = range(1, len(history.history["loss"]) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor("#0a150a")

    for ax in (ax1, ax2):
        ax.set_facecolor("#0f1f0f")
        ax.tick_params(colors="white")
        ax.xaxis.label.set_color("white")
        ax.yaxis.label.set_color("white")
        ax.title.set_color("white")
        ax.spines[:].set_color("#1a3d1a")

    # Loss
    ax1.plot(epochs, history.history["loss"],     color="#7ec850", lw=2, label="Train Loss")
    ax1.plot(epochs, history.history["val_loss"], color="#ff8c00", lw=2, linestyle="--", label="Val Loss")
    ax1.set_title(f"Loss — {phase_label}")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.legend(labelcolor="white", framealpha=0.2)
    ax1.grid(alpha=0.2, color="#1a3d1a")

    # Accuracy
    ax2.plot(epochs, history.history["accuracy"],     color="#7ec850", lw=2, label="Train Acc")
    ax2.plot(epochs, history.history["val_accuracy"], color="#ff8c00", lw=2, linestyle="--", label="Val Acc")
    ax2.set_title(f"Accuracy — {phase_label}")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
    ax2.legend(labelcolor="white", framealpha=0.2)
    ax2.grid(alpha=0.2, color="#1a3d1a")

    plt.suptitle(f"KrishiAI Training — {phase_label}", color="white", fontsize=13, y=1.01)
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Training curve saved → {path}")


# ──────────────────────────────────────────────────────────────────────────────
# MAIN TRAINING LOOP
# ──────────────────────────────────────────────────────────────────────────────

def train():
    print("╔══════════════════════════════════════════════════════════╗")
    print("║        KrishiAI — Model Training (Step 2 of 4)          ║")
    print("║     EfficientNetB0 Transfer Learning — Two Phases       ║")
    print("╚══════════════════════════════════════════════════════════╝")

    # ── Load data ─────────────────────────────────────────────────────────────
    print("\n[1/2] Loading datasets from manifests...")
    train_ds, val_ds, test_ds = load_datasets_from_manifests()
    class_weights = load_class_weights()

    # ── PHASE 1: Train the head only ──────────────────────────────────────────
    print("\n" + "="*60)
    print("PHASE 1 — Training Classification Head (base frozen)")
    print("="*60)
    print(f"  Epochs: {EPOCHS_FROZEN}  LR: {LEARNING_RATE_HEAD}")

    model, base = build_model(trainable_base=False)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_HEAD),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.TopKCategoricalAccuracy(k=2, name="top2_accuracy"),
            keras.metrics.AUC(name="auc", multi_label=False),
        ]
    )
    model.summary()

    print(f"\n  Trainable params: {model.count_params():,}")
    callbacks_p1, ckpt_p1 = get_callbacks(phase=1)
    t0 = time.time()
    hist1 = model.fit(
        train_ds,
        epochs=EPOCHS_FROZEN,
        validation_data=val_ds,
        callbacks=callbacks_p1,
        class_weight=class_weights,
        verbose=1,
    )
    t1 = time.time()
    print(f"\n  Phase 1 complete in {(t1-t0)/60:.1f} min")
    print(f"  Best val accuracy: {max(hist1.history['val_accuracy']):.4f}")
    plot_history(hist1, "Phase 1 — Head Only", "phase1_training_curves.png")

    # Load best Phase 1 checkpoint
    print(f"\n  Loading best Phase 1 checkpoint: {ckpt_p1}")
    model = keras.models.load_model(ckpt_p1)

    # ── PHASE 2: Fine-tune top layers ─────────────────────────────────────────
    print("\n" + "="*60)
    print("PHASE 2 — Fine-tuning Top Layers (partial unfreeze)")
    print("="*60)
    print(f"  Epochs: {EPOCHS_FINETUNE}  LR: {LEARNING_RATE_FINETUNE}")

    # Unfreeze top 30 layers of the EfficientNetB0 base
    base_layer = model.get_layer("efficientnetb0")
    base_layer.trainable = True
    for layer in base_layer.layers[:-30]:
        layer.trainable = False

    # Re-compile with much smaller LR to avoid catastrophic forgetting
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_FINETUNE),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.TopKCategoricalAccuracy(k=2, name="top2_accuracy"),
            keras.metrics.AUC(name="auc", multi_label=False),
        ]
    )

    callbacks_p2, ckpt_p2 = get_callbacks(phase=2)
    t2 = time.time()
    hist2 = model.fit(
        train_ds,
        epochs=EPOCHS_FINETUNE,
        validation_data=val_ds,
        callbacks=callbacks_p2,
        class_weight=class_weights,
        verbose=1,
    )
    t3 = time.time()
    print(f"\n  Phase 2 complete in {(t3-t2)/60:.1f} min")
    print(f"  Best val accuracy: {max(hist2.history['val_accuracy']):.4f}")
    plot_history(hist2, "Phase 2 — Fine-tuning", "phase2_training_curves.png")

    # ── Save final model ──────────────────────────────────────────────────────
    final_model = keras.models.load_model(ckpt_p2)
    final_path  = os.path.join(MODELS_DIR, "krishiai_final.keras")
    final_model.save(final_path)
    print(f"\n  Final model saved → {final_path}")

    # ── Quick evaluation on val set ───────────────────────────────────────────
    print("\n  Quick evaluation on validation set:")
    val_results = final_model.evaluate(val_ds, verbose=0)
    for name, val in zip(final_model.metrics_names, val_results):
        print(f"    {name:<25} {val:.4f}")

    # Save combined history
    all_history = {
        "phase1_accuracy": hist1.history["accuracy"],
        "phase1_val_accuracy": hist1.history["val_accuracy"],
        "phase1_loss": hist1.history["loss"],
        "phase1_val_loss": hist1.history["val_loss"],
        "phase2_accuracy": hist2.history["accuracy"],
        "phase2_val_accuracy": hist2.history["val_accuracy"],
        "phase2_loss": hist2.history["loss"],
        "phase2_val_loss": hist2.history["val_loss"],
    }
    with open(os.path.join(LOGS_DIR, "training_history.json"), "w") as f:
        json.dump(all_history, f)

    print(f"\n✅ Training complete!")
    print(f"   Final model → {final_path}")
    print(f"\nNext step: python step3_evaluate.py")
    return final_model



## ▶️ Phase 1 — Train Classification Head (Base Frozen)

In [ ]:
# Load datasets from manifests saved by step1
print("Loading datasets...")
train_ds, val_ds, test_ds = load_datasets_from_manifests()
class_weights = load_class_weights()
print(f"\nClass weights: {class_weights}")

In [ ]:
# Build model with frozen base
model, base = build_model(trainable_base=False)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_HEAD),
    loss="categorical_crossentropy",
    metrics=["accuracy",
             keras.metrics.TopKCategoricalAccuracy(k=2, name="top2_accuracy"),
             keras.metrics.AUC(name="auc", multi_label=False)]
)
model.summary()

In [ ]:
callbacks_p1, ckpt_p1 = get_callbacks(phase=1)

print("Starting Phase 1 training...")
t0 = time.time()
hist1 = model.fit(
    train_ds,
    epochs=EPOCHS_FROZEN,
    validation_data=val_ds,
    callbacks=callbacks_p1,
    class_weight=class_weights,
    verbose=1,
)
print(f"\nPhase 1 done in {(time.time()-t0)/60:.1f} min")
print(f"Best val accuracy: {max(hist1.history['val_accuracy']):.4f}")
plot_history(hist1, "Phase 1 — Head Only", "phase1_training_curves.png")

In [ ]:
from IPython.display import Image as IPyImage, display
display(IPyImage(os.path.join(PLOTS_DIR, "phase1_training_curves.png")))

## ▶️ Phase 2 — Fine-Tune Top 30 Layers

In [ ]:
# Load best Phase 1 checkpoint, then unfreeze top 30 layers
model = keras.models.load_model(ckpt_p1)

base_layer = model.get_layer("efficientnetb0")
base_layer.trainable = True
for layer in base_layer.layers[:-30]:
    layer.trainable = False

trainable_count = sum(1 for l in base_layer.layers if l.trainable)
print(f"Unfrozen layers in base: {trainable_count} / {len(base_layer.layers)}")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_FINETUNE),
    loss="categorical_crossentropy",
    metrics=["accuracy",
             keras.metrics.TopKCategoricalAccuracy(k=2, name="top2_accuracy"),
             keras.metrics.AUC(name="auc", multi_label=False)]
)

In [ ]:
callbacks_p2, ckpt_p2 = get_callbacks(phase=2)

print("Starting Phase 2 fine-tuning...")
t2 = time.time()
hist2 = model.fit(
    train_ds,
    epochs=EPOCHS_FINETUNE,
    validation_data=val_ds,
    callbacks=callbacks_p2,
    class_weight=class_weights,
    verbose=1,
)
print(f"\nPhase 2 done in {(time.time()-t2)/60:.1f} min")
print(f"Best val accuracy: {max(hist2.history['val_accuracy']):.4f}")
plot_history(hist2, "Phase 2 — Fine-tuning", "phase2_training_curves.png")

In [ ]:
from IPython.display import Image as IPyImage, display
display(IPyImage(os.path.join(PLOTS_DIR, "phase2_training_curves.png")))

## 💾 Save Final Model

In [ ]:
final_model = keras.models.load_model(ckpt_p2)
final_path  = os.path.join(MODELS_DIR, "krishiai_final.keras")
final_model.save(final_path)
print(f"✅ Final model saved → {final_path}")
print(f"   Size: {os.path.getsize(final_path)/1e6:.1f} MB")

print("\nValidation set evaluation:")
val_results = final_model.evaluate(val_ds, verbose=1)
for name, val in zip(final_model.metrics_names, val_results):
    print(f"  {name:<25} {val:.4f}")

In [ ]:
# Save combined training history
all_history = {
    "phase1_accuracy":     hist1.history["accuracy"],
    "phase1_val_accuracy": hist1.history["val_accuracy"],
    "phase1_loss":         hist1.history["loss"],
    "phase1_val_loss":     hist1.history["val_loss"],
    "phase2_accuracy":     hist2.history["accuracy"],
    "phase2_val_accuracy": hist2.history["val_accuracy"],
    "phase2_loss":         hist2.history["loss"],
    "phase2_val_loss":     hist2.history["val_loss"],
}
with open(os.path.join(LOGS_DIR, "training_history.json"), "w") as f:
    json.dump(all_history, f)
print(f"Training history saved → {LOGS_DIR}/training_history.json")
print("\nNext: open 04_step3_evaluate.ipynb")